# W4A16 dequant-fused matmul kernel — Colab walkthrough

4-bit group-wise quantized weights + fp16 activations, with a fused dequant→GEMM Triton kernel (primary) and an optional CUDA SIMT kernel (stretch).

**Before running:** `Runtime > Change runtime type > T4 GPU`, and make sure the `w4a16-kernel` folder is present in this Colab session (upload it, mount Drive, or `!git clone` your repo).

Runs top-to-bottom: setup → quant demo → correctness → benchmarks → plots → results table → (stretch) CUDA.

## M0 — setup & capability probe

In [ ]:
# Colab ships torch + triton already; install only the light extras.
!pip install -q numpy matplotlib pytest

In [ ]:
import sys, os

# Locate the repo and put src/ on the path.
CANDIDATES = ['w4a16-kernel', '.', '/content/w4a16-kernel']
SRC = None
for base in CANDIDATES:
    p = os.path.join(base, 'src', 'w4a16', '__init__.py')
    if os.path.exists(p):
        SRC = os.path.abspath(os.path.join(base, 'src'))
        break
assert SRC, 'Could not find w4a16-kernel/src — upload or git clone the repo into Colab first.'
sys.path.insert(0, SRC)
REPO = os.path.dirname(SRC)
print('repo:', REPO)

import torch
print('torch :', torch.__version__, '| cuda build:', torch.version.cuda)
try:
    import triton; print('triton:', triton.__version__)
except Exception as e:
    print('triton import failed:', e)

assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU.'
print('device:', torch.cuda.get_device_name(0))
print('capability:', torch.cuda.get_device_capability(0))
print('bf16 supported:', torch.cuda.is_bf16_supported(), '(T4 = False; we use fp16 + fp32 accumulate)')

## M1 — quantization + packing demo

In [ ]:
import torch
from w4a16 import quantize_weight, dequantize_weight, pack_int4, unpack_int4

torch.manual_seed(0)
K, N, G = 4096, 4096, 128
W = torch.randn(K, N, dtype=torch.float16, device='cuda')
qweight, scales, zeros = quantize_weight(W, G)
print('qweight:', tuple(qweight.shape), qweight.dtype, '(packed int4, ~4x smaller than fp16 W)')
print('scales :', tuple(scales.shape), scales.dtype)
print('zeros  :', tuple(zeros.shape), zeros.dtype)

# pack/unpack roundtrip on random uint4
r = torch.randint(0, 16, (256, 64), dtype=torch.int32, device='cuda')
assert torch.equal(unpack_int4(pack_int4(r)), r)
print('pack/unpack roundtrip: OK')

# quantize -> dequantize error
W_dq = dequantize_weight(qweight, scales, zeros, G)
rel = (W_dq.float() - W.float()).abs().mean() / W.float().abs().mean()
print(f'mean relative quant error: {rel.item()*100:.2f}%')

fp16_bytes = K*N*2
packed_bytes = qweight.numel()*4 + scales.numel()*2 + zeros.numel()*2
print(f'weight bytes: fp16={fp16_bytes/1e6:.1f} MB  packed={packed_bytes/1e6:.1f} MB  ({fp16_bytes/packed_bytes:.2f}x)')

## M3 — correctness: fused Triton kernel == fp32-accumulate oracle

Both sides use the SAME (qweight, scales, zeros), isolating kernel correctness from quantization error. First launch triggers autotune (6 configs).

In [ ]:
import torch
from w4a16 import quantize_weight, reference_w4a16, w4a16_matmul

shapes = [(1,4096,4096),(16,4096,11008),(64,11008,4096),(256,4096,4096)]
for (M,K,N) in shapes:
    for G in (64,128):
        X = torch.randn(M,K,dtype=torch.float16,device='cuda')
        W = torch.randn(K,N,dtype=torch.float16,device='cuda')
        qw,s,z = quantize_weight(W,G)
        ref = reference_w4a16(X,qw,s,z,G)
        out = w4a16_matmul(X,qw,s,z,G)
        err = (out.float()-ref.float()).abs().max().item()
        ok = torch.allclose(out,ref,rtol=1e-2,atol=1e-2)
        print(f'M={M:<4} K={K:<6} N={N:<6} G={G:<3} max_abs_err={err:.4f}  {"PASS" if ok else "FAIL"}')
        assert ok, 'kernel != reference'
print('\nAll inline correctness checks passed.')

In [ ]:
# (Optional) full test suite. Takes a few minutes (autotune over all shapes).
!cd {REPO} && python -m pytest tests/ -q

## M4 — benchmarks: latency / TFLOPS / GB/s vs baselines

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(REPO, 'benchmarks'))
from bench_gemm import run_sweep, to_markdown_table

results = run_sweep(group_size=128)
print()
print(to_markdown_table(results))

In [ ]:
from plot_results import plot_latency, plot_speedup, append_table_to_readme
from IPython.display import Image, display
import os

out_dir = os.path.join(REPO, 'benchmarks', 'plots')
os.makedirs(out_dir, exist_ok=True)
plot_latency(results, os.path.join(out_dir, 'latency.png'))
plot_speedup(results, os.path.join(out_dir, 'speedup.png'))
append_table_to_readme(results)
display(Image(os.path.join(out_dir, 'latency.png')))
display(Image(os.path.join(out_dir, 'speedup.png')))

## M4 — `W4A16Linear` drop-in demo

In [ ]:
import torch
from w4a16 import W4A16Linear

lin = torch.nn.Linear(4096, 4096, bias=True).cuda().half()
qlin = W4A16Linear.from_linear(lin, group_size=128)
x = torch.randn(1, 4096, dtype=torch.float16, device='cuda')
y_fp16 = lin(x)
y_w4a16 = qlin(x)
print('fp16 vs W4A16 max abs diff:', (y_fp16.float() - y_w4a16.float()).abs().max().item())
print('(nonzero = 4-bit quantization error, expected)')

## M5 — CUDA SIMT kernel (stretch)

Compiles `cuda/w4a16_gemm.cu` for the detected arch. First run takes ~1 minute.

In [ ]:
import torch
from w4a16.cuda_kernel import load_cuda_module, w4a16_matmul_cuda
from w4a16 import quantize_weight, reference_w4a16

load_cuda_module()  # compiles + caches
M,K,N,G = 64,4096,4096,128
X = torch.randn(M,K,dtype=torch.float16,device='cuda')
W = torch.randn(K,N,dtype=torch.float16,device='cuda')
qw,s,z = quantize_weight(W,G)
ref = reference_w4a16(X,qw,s,z,G)
out = w4a16_matmul_cuda(X,qw,s,z,G)
print('CUDA max abs err:', (out.float()-ref.float()).abs().max().item())
print('allclose:', torch.allclose(out,ref,rtol=1e-2,atol=1e-2))